# 01 — EDA: clinical (synthetic cohort)

Distributional checks on tabular clinical features produced by
`generate_synthetic_cohort`. Run after `pip install -e .` from the repo root
so `mmvlm4scd` is importable.


## Setup


In [ ]:
import sys
from pathlib import Path

# Allow running from notebooks/ without editable install (optional)
_repo = Path.cwd()
for _ in range(4):
    if (_repo / "src" / "mmvlm4scd").is_dir():
        sys.path.insert(0, str(_repo / "src"))
        break
    _repo = _repo.parent

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mmvlm4scd.data import StandardPreprocessor, generate_synthetic_cohort
from mmvlm4scd.data.synthetic import SCDSyntheticConfig


## Load cohort and preprocess


In [ ]:
cfg = SCDSyntheticConfig(n_patients=1200, seed=42)
cohort = generate_synthetic_cohort(cfg)
clin: pd.DataFrame = cohort["clinical"]
clin.head()


In [ ]:
pre = StandardPreprocessor().fit(clin)
x_clin = pre.transform(clin)
print("Preprocessor output_dim (matches MultimodalSCDModel clinical_input_dim):", pre.output_dim)
print("Feature matrix shape:", x_clin.shape)


## Summary statistics


In [ ]:
clin.describe(include="all").T


## Genotype mix (Hb phenotypes)


In [ ]:
vc = clin["genotype"].value_counts(normalize=True).sort_index()
display(vc)
vc.plot(kind="bar", title="Genotype prevalence (synthetic)", rot=45)
plt.ylabel("fraction")
plt.tight_layout()
plt.show()


## Labs vs severity label


In [ ]:
sev = cohort["severity"]
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, col in zip(axes, ["hb_g_dl", "ldh_u_l", "voc_rate_per_year"]):
    for k in range(3):
        mask = sev == k
        ax.hist(clin.loc[mask, col].values, bins=20, alpha=0.45, label=f"sev {k}")
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
